###  The Date Parsing Workflow

1. **Identify**: Inspect the raw CSV to see the exact character sequence of the date (e.g., `2020-03-13 08-PM`).
2. **Map**: Create a `date_format` string that mirrors the input exactly.
   * `%Y` for 2020, `%y` for 20.
   * `%m` for month, `%d` for day.
   * `%I` for 12-hour clock, `%p` for AM/PM.
3. **Load**: Use `pd.read_csv(..., parse_dates=['Date'], date_format='...')` to convert strings into functional `datetime64` objects.
4. **Transform (Optional)**: After loading, use `.dt.strftime()` only if you need to display the date in a different visual format for reports (e.g., `13-March-2020`).

---

###  Pro-Level Time Operations

* **Indexing**: Always set the date as your index using `df.set_index('Date', inplace=True)`. This unlocks intuitive slicing:
  * `df['2020']` -> Fetches all data for the year 2020.
  * `df['2020-01':'2020-02']` -> Fetches data for January and February.
* **Resampling**: Use `.resample()` to change data frequency (e.g., from Hourly to Weekly).
  * `df['High'].resample('W').max()` -> The highest price of each week.
* **Feature Engineering**:
  * `.dt.day_name()`: Extract the day of the week to check for weekend patterns.
  * `.shift(1)`: Create "Lag Features" (today's row shows yesterday's price).
  * `.rolling(window=7).mean()`: Calculate moving averages to see the overall trend.

---

###  Engineering Rule:
> **"The `date_format` is a translator, not a formatter. It tells Pandas how to READ the file, not how to SHOW it."**

In [ ]:
import pandas as pd

df = pd.read_csv("ETH_1h.csv", parse_dates=["Date"], date_format="%Y-%m-%d %I-%p")


In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.loc[0, "Date"]

In [ ]:
# df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d %I-%p")

In [ ]:
df["Date"]

In [ ]:
df.loc[0, "Date"].day_name()

In [ ]:
df.loc[70, "Date"].day_name()

In [ ]:
df["Date"].dt.day_name()

In [ ]:
df["DayOfWeek"] = df["Date"].dt.day_name()

In [ ]:
df

In [ ]:
df["Date"].min()

In [ ]:
df["Date"].max()

In [ ]:
df["Date"].max() - df["Date"].min()

In [ ]:
mask = ((df["Date"] >= pd.to_datetime("2019-08-01")) & (df["Date"] <= pd.to_datetime("2019-09-01")))
df.loc[mask]

In [ ]:
df.set_index("Date", inplace=True)


In [ ]:
df.loc["2019"]

In [ ]:
df.sort_index(inplace=True)

In [ ]:
df.loc["2019-01":"2019-02"]

In [ ]:
df.loc["2019-01":"2019-02"]["Close"]

In [ ]:
df.loc["2020-01":"2020-02"]["Close"].mean()

In [ ]:
df.loc["2020-01":"2020-02"].head(24)

In [ ]:
df.loc["2020-01-01"]["High"].max()
# print(df.loc["2020-01-01"]["High"].max())

In [ ]:
highs = df["High"].resample("D").max()

In [ ]:
highs["2020-01-01"]

In [ ]:
%matplotlib inline

In [ ]:
highs.plot()

In [ ]:

df.resample("W").mean(numeric_only=True)

In [ ]:
df.resample("W").agg({"Close": "mean", "High": "max", "Low": "min", "Volume": "sum"})

In [ ]:
df["Date"].shift(1)

In [ ]:

df['Time_Delta'] = df['Date'] - df['Date'].shift(1)

In [ ]:
df

In [ ]:

df["Close"].rolling(window=7).std()